# 第 33 章 · 注册自定义算子：让你的 kernel 成为 PyTorch 的"一等公民"

本章回答：自己写了一个融合 kernel，直接在 Python 里调用有什么问题？`torch.library.custom_op` 注册后得到了什么？`register_fake`、`register_autograd` 分别给谁用？`opcheck` 查什么？Triton kernel 为什么推荐用 `triton_op`？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/33-custom-op.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch14「向 PyTorch 注册自定义内核」（`ch14.md` 第 1257–1323 行）；官方教程「Python Custom Operators」与 `torch.library` 文档（〔在线〕docs.pytorch.org/tutorials/advanced/python_custom_ops.html）；本讲义 [第 3 章 分派器](03-dispatcher.md)、[第 4 章 Autograd](04-autograd.md)、[第 6 章 compile](06-compile-stack.md)；本章所有输出在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
from torch import Tensor
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 没有 fake | 编译报错 | `compile_without_fake()` | TorchRuntimeError |
| 第二步 有 fake | 不断图 | `graph_breaks()` | (1, 0) |
| 第三步 反向 | 上游 × [out > 0] | `forward_backward()` | 输出 [0, 0, 4]，两个梯度 [0, 0, 1] |
| 第四步 opcheck | 四项 | `opcheck()` | 全部 SUCCESS |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `times2_nofake`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
@torch.library.custom_op("ptdd::times2_nofake", mutates_args=())
def times2_nofake(x: Tensor) -> Tensor:
    return x * 2

**函数 `compile_without_fake`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
def compile_without_fake() -> str:
    torch._dynamo.reset()
    try:
        torch.compile(lambda a: times2_nofake(a) + 1, fullgraph=True)(torch.randn(3))
        return "编译成功"
    except Exception as e:
        return type(e).__name__

**函数 `add_relu`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
@torch.library.custom_op("ptdd::add_relu", mutates_args=())
def add_relu(x: Tensor, y: Tensor) -> Tensor:
    return torch.relu(x + y)          # 真正的实现：可以换成 Triton / C++ / 任何东西

**函数 `_`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
@add_relu.register_fake
def _(x, y):
    return torch.empty_like(x)

**函数 `_backward`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
def _backward(ctx, grad):
    (out,) = ctx.saved_tensors
    g = grad * (out > 0)
    return g, g

**函数 `_setup_context`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
def _setup_context(ctx, inputs, output):
    ctx.save_for_backward(output)

**登记**（来自 `ch33_custom_op.py`）：把上面的函数挂到算子上。

In [ ]:
add_relu.register_autograd(_backward, setup_context=_setup_context)

**函数 `forward_backward`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
def forward_backward():
    x = torch.tensor([-2.0, 1.0, 3.0], requires_grad=True)
    y = torch.tensor([1.0, -3.0, 1.0], requires_grad=True)
    out = add_relu(x, y)
    out.sum().backward()
    return out.tolist(), x.grad.tolist(), y.grad.tolist(), type(out.grad_fn).__name__

**函数 `graph_breaks`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
def graph_breaks():
    def f(a, b):
        return add_relu(a * 2, b).sin()

    e = torch._dynamo.explain(f)(torch.randn(4), torch.randn(4))
    return e.graph_count, e.graph_break_count

**函数 `opcheck`**（来自 `ch33_custom_op.py`）：下面要用到的小工具。

In [ ]:
def opcheck():
    return torch.library.opcheck(add_relu, (torch.randn(4, requires_grad=True), torch.randn(4, requires_grad=True)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 没有 fake 的算子去编译：", compile_without_fake())
print("有 fake 的算子在图里：(图数, 断图数) =", graph_breaks())

In [ ]:
print("\n第三步 前向、两个输入的梯度、grad_fn：", forward_backward())

In [ ]:
print("\n第四步 opcheck：", opcheck())

## 3. 代码里才看得到的

1. **命名空间用了 `ptdd`**：讲义里写 `mylib`，代码里换成本专题的缩写，避免和别处注册的同名算子冲突。
2. **同一进程里不能重复注册同名算子**：要改实现就换个名字或重启 Python。

## 4. 自己改一改

- 故意把 fake 写成 `torch.empty(1)`（形状错），看 opcheck 哪一项失败。
- 去掉 `register_autograd`，对输出 `.sum().backward()` 看报什么。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 直接调用没人认，注册进分派表才算正式算子；
> custom_op 写签名、声明改了谁，register_fake 只报形状，compile、export 才能用；
> register_autograd 存东西给梯度；opcheck 查注册，assert_close 查数值；Triton 用 triton_op，编译器看得进去。